# Day 6: 两块小数字理解 Online Softmax

配套讲义：`docs/DAY06_ONLINE_SOFTMAX.md`。使用CPU，先推导数学，不测试GPU速度。
目标不是背公式，而是理解旧累积量为什么要换基准。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/online_attention.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from torch.nn import functional as F
from learning.online_attention import online_softmax_chunks, blocked_attention
scores = torch.tensor([1., 2., 3., 4.], dtype=torch.float64)
values = torch.tensor([10., 20., 30., 40.], dtype=torch.float64)

## 1. 正确答案与一种错误合并法

先计算完整softmax，再把两个块分别softmax、求输出、简单平均。
为什么简单平均错误？因为它强行让两块获得相同总权重。

In [ ]:
reference = scores.softmax(0) @ values
first = scores[:2].softmax(0) @ values[:2]
second = scores[2:].softmax(0) @ values[2:]
incorrect = (first + second) / 2
print('正确完整输出:', reference.item())
print('第1块自己的输出:', first.item())
print('第2块自己的输出:', second.item())
print('错误简单平均:', incorrect.item())
assert not torch.isclose(reference, incorrect)

## 2. 维护 m、l、o

`m`是见过的最大分数，`l`是当前基准下的指数和，`o`是加权Value和。
每次最大值变大，旧l/o同时乘 `exp(m_old-m_new)`，再加入新块。

In [ ]:
m = scores.new_tensor(-torch.inf)
denominator = scores.new_zeros(())
numerator = scores.new_zeros(())
for start in (0, 2):
    block_s, block_v = scores[start:start+2], values[start:start+2]
    new_m = torch.maximum(m, block_s.max())
    alpha = torch.exp(m - new_m)
    p = torch.exp(block_s - new_m)
    denominator = alpha * denominator + p.sum()
    numerator = alpha * numerator + p @ block_v
    m = new_m
    print(f'处理位置 {start}..{start+1}: m={m.item():.4f}, alpha={alpha.item():.4f}, l={denominator.item():.4f}, o={numerator.item():.4f}')
actual = numerator / denominator
print('Online结果:', actual.item())
torch.testing.assert_close(actual, reference)
print('与完整softmax对齐: passed')

## 3. 改块宽与大幅值输入

不要只测一个切分，也不要只测不会溢出的小分数。

In [ ]:
for chunk in (1, 2, 3, 8):
    probabilities, history = online_softmax_chunks(scores, chunk)
    torch.testing.assert_close(probabilities, scores.softmax(0))
    print('chunk:', chunk, 'output:', (probabilities @ values).item())
large = scores + 1000
stable, _ = online_softmax_chunks(large, 2)
torch.testing.assert_close(stable, scores.softmax(0))
assert torch.isfinite(stable).all()
print('整体增加1000后，稳定结果不变；直接exp可能溢出:', torch.isfinite(large.exp()).all().item())

## 4. 从一行分数回到Attention

真实Attention的分子o是向量；下面复用课程中的分块PyTorch参考，对照SDPA。
这不是生产GPU FlashAttention kernel，也不包含训练backward验收。

In [ ]:
torch.manual_seed(123)
q, k, v = [torch.randn(1, 2, 7, 8) for _ in range(3)]
expected = F.scaled_dot_product_attention(q, k, v, is_causal=True)
for q_block, k_block in ((1, 1), (3, 2), (4, 3)):
    result = blocked_attention(q, k, v, query_block=q_block, key_block=k_block)
    torch.testing.assert_close(result, expected, atol=1e-5, rtol=1e-5)
    print('Q/K block:', q_block, k_block, '最大误差:', (result - expected).abs().max().item())

## 自测

- 每块softmax后简单平均，丢掉了什么信息？
- 为什么旧分子和旧分母必须一起乘alpha？
- 分块避免保存score，是否意味着KV Cache也不需要了？
- 为什么这份数学参考通过，并不证明生产GPU性能或训练backward？

继续运行 `learning/day06_course.ipynb` 保存统一报告。个人验收后再进入Day 7。